## A6b - Coronal-hole catalog ingestion and 3h grid

This notebook downloads coronal-hole detections and aggregates them to a fixed 3h cadence. The output is used as solar-origin features for the extension experiments.

### Data source and download
The catalog is queried from the SPoCA coronal-hole dataset via the ROB/OMA TAP service:
https://vo-tap.oma.be/tap

Query table: `rob_spoca_ch.epn_core`

Fields used:
- time_min, granule_gid
- ch_area_projected, ch_area_deprojected
- ch_c1_centroid, ch_c2_centroid
- ch_stat_hmi_mean, ch_stat_hmi_variance
- subobserver_longitude_min

Coverage in this notebook: 2010 to 2025 (inclusive).

### Outputs
- Raw yearly slices: `Data/processed/solar_origin_raw/spoca_raw_YYYY.parquet`
- 3h grid features: `Data/processed/solar_origin_ch_features.parquet`


In [15]:
import pandas as pd
from pyvo.dal import tap
from astropy.time import Time
import warnings
from astropy.units import UnitsWarning
from pathlib import Path

# --- Config ---
START_YEAR = 2010
END_YEAR = 2025  # inclusive
OUT_RAW_DIR = '../Data/processed/solar_origin_raw'
OUT_6H_PATH = '../Data/processed/solar_origin_ch_features.parquet'

warnings.filterwarnings('ignore', category=UnitsWarning)
service = tap.TAPService('https://vo-tap.oma.be/tap')

query_tpl = (
    'SELECT time_min, granule_gid,\n'
    '       ch_area_projected, ch_area_deprojected,\n'
    '       ch_c1_centroid, ch_c2_centroid,\n'
    '       ch_stat_hmi_mean, ch_stat_hmi_variance,\n'
    '       subobserver_longitude_min\n'
    'FROM rob_spoca_ch.epn_core\n'
    'WHERE time_min >= {start_jd} AND time_min < {end_jd}\n'
    'ORDER BY time_min'
)

raw_frames = []
Path(OUT_RAW_DIR).mkdir(parents=True, exist_ok=True)

for year in range(START_YEAR, END_YEAR + 1):
    start_jd = Time(f'{year}-01-01').jd
    end_jd = Time(f'{year + 1}-01-01').jd
    query = query_tpl.format(start_jd=start_jd, end_jd=end_jd)

    print(f'Fetching SPoCA {year}...')
    results = service.run_sync(query)
    df = results.to_table().to_pandas()
    if df.empty:
        print(f'  {year}: no rows')
        continue
    df['timestamp_utc'] = pd.to_datetime(Time(df['time_min'], format='jd').iso)

    # Save raw yearly slice (handy if you need to re-merge without re-downloading)
    out_raw = f"{OUT_RAW_DIR}/spoca_raw_{year}.parquet"
    df.to_parquet(out_raw)
    print(f'  {year}: {len(df)} rows -> {out_raw}')

    raw_frames.append(df)

if not raw_frames:
    raise SystemExit('No SPoCA rows fetched. Check dates / API access.')

df_all = pd.concat(raw_frames, ignore_index=True)
df_all = df_all.sort_values('timestamp_utc').reset_index(drop=True)

# Persistence: count consecutive gid per window (recurrence proxy)
df_all['gid_count'] = df_all.groupby('granule_gid').cumcount() + 1

# 3h agg: sum areas, mean centroids/long/B, max gid_count
df_3h = (df_all.set_index('timestamp_utc')
         .resample('3h')
         .agg({
    'ch_area_deprojected': 'sum',
    'ch_area_projected': 'sum',
    'ch_c2_centroid': 'mean',
    'ch_c1_centroid': 'mean',
    'subobserver_longitude_min': 'mean',
    'ch_stat_hmi_mean': 'mean',
    'ch_stat_hmi_variance': 'mean',
    'gid_count': 'max'
})
         .reset_index(drop=False)
         .rename(columns={
    'ch_area_deprojected': 'coronal_hole_area_total',
    'ch_c2_centroid': 'coronal_hole_lat_mean',
    'ch_c1_centroid': 'coronal_hole_lon_mean',
    'subobserver_longitude_min': 'central_meridian_distance',
    'ch_stat_hmi_mean': 'coronal_hole_polarity_balance',
    'gid_count': 'coronal_hole_persistence_days'
})
         .sort_values('timestamp_utc')
         .reset_index(drop=True)
         )

# NaN → 0 for sums, forward-fill means
df_3h = df_3h.fillna(
    {'coronal_hole_area_total': 0, 'coronal_hole_persistence_days': 0}
).ffill()

df_3h.to_parquet(OUT_6H_PATH)
print(f"Enhanced grid: {len(df_3h)} rows -> {OUT_6H_PATH}")
print(df_3h.head())

print("\nPhysics: area_total → stream strength; lat/lon → geoeff.; pol_balance → IMF; persistence → recurrence.")



Fetching SPoCA 2010...
  2010: 4657 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2010.parquet
Fetching SPoCA 2011...
  2011: 4524 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2011.parquet
Fetching SPoCA 2012...
  2012: 4097 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2012.parquet
Fetching SPoCA 2013...
  2013: 3538 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2013.parquet
Fetching SPoCA 2014...
  2014: 3535 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2014.parquet
Fetching SPoCA 2015...
  2015: 5539 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2015.parquet
Fetching SPoCA 2016...
  2016: 6580 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2016.parquet
Fetching SPoCA 2017...
  2017: 6468 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2017.parquet
Fetching SPoCA 2018...
  2018: 6855 rows -> ../Data/processed/solar_origin_raw/spoca_raw_2018.parquet
Fetching SPoCA 2019...
  2019: 6457 rows -> ../Data/processed/solar_origin_raw/spo

In [17]:
# Reload cached 3h grid (skip API download)
import pandas as pd
from pathlib import Path

CACHED_PATH = Path('../Data/processed/solar_origin_features.parquet')
if not CACHED_PATH.exists():
    raise FileNotFoundError(f'Cannot find {CACHED_PATH}')

df_3h = pd.read_parquet(CACHED_PATH)
print(f'Enhanced grid: {len(df_3h)} rows -> {CACHED_PATH}')
print(df_3h.head())


Enhanced grid: 23252 rows -> ../Data/processed/solar_origin_features.parquet
              timestamp_utc  cme_speed_mean  cme_speed_max  cme_width_mean  \
0 2010-01-01 06:00:00+00:00           239.0          330.0            40.5   
1 2010-01-01 12:00:00+00:00           729.0          729.0            21.0   
2 2010-01-01 18:00:00+00:00             0.0            0.0             0.0   
3 2010-01-02 00:00:00+00:00             0.0            0.0             0.0   
4 2010-01-02 06:00:00+00:00           141.0          141.0            14.0   

   cme_width_max  cme_cpa_mean  cme_mpa_mean  cme_count  cme_halo_count  \
0           75.0         301.0         304.0          2               0   
1           21.0          49.0          52.0          1               0   
2            0.0           0.0           0.0          0               0   
3            0.0           0.0           0.0          0               0   
4           14.0         100.0         106.0          1               0   

   